# Q1 — M-Net Brain Tissue Segmentation (scaffold)

This notebook will contain a full, self-contained implementation of Q1 (M-Net) with all helper functions defined inline (no external .py imports). Steps:

- Data preparation (read .nii.gz, extract 2D slices, normalization, save .npy)
- Dataset and DataLoader with collate_fn
- M-Net model definition (PyTorch)
- Training loop, metrics (Dice, Jaccard, Precision, Recall) implemented from scratch
- Plots and visualizations


In [ ]:
# Environment & imports
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from typing import Tuple, List

In [ ]:
# Data prep helpers (placeholders — will implement full functions below)
def load_nifti(path: str) -> np.ndarray:
    """Load a .nii or .nii.gz using nibabel — implemented inline in full version."""
    raise NotImplementedError


class IBSR2DSlicesDataset(Dataset):
    def __init__(self, data_paths: List[str]):
        raise NotImplementedError


def collate_pad(batch):
    raise NotImplementedError

## Model skeleton

We'll implement M-Net (U-Net variant) in PyTorch below.


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class MNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=4, base_filters=32):
        super().__init__()
        # Encoder
        self.enc1 = DoubleConv(in_channels, base_filters)
        self.pool = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(base_filters, base_filters * 2)
        # Bottleneck and decoder (skeleton)
        self.up = nn.ConvTranspose2d(base_filters * 2, base_filters, 2, stride=2)
        self.dec1 = DoubleConv(base_filters * 2, base_filters)
        self.final = nn.Conv2d(base_filters, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        d1 = self.up(e2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        return self.final(d1)


# quick model instantiation
model = MNet()
model

In [ ]:
# Full data-prep and dataset implementation
import nibabel as nib
import glob
from pathlib import Path


def load_nifti(path: str) -> np.ndarray:
    """Load a .nii or .nii.gz file and return numpy array (float32)."""
    img = nib.load(path)
    data = img.get_fdata()
    return np.asarray(data, dtype=np.float32)


def map_labels(mask: np.ndarray) -> np.ndarray:
    """Map raw IBSR labels to 0:background,1:CSF,2:GM,3:WM.
    Adjust mapping if your dataset uses different label ids.
    """
    out = np.zeros_like(mask, dtype=np.uint8)
    # Heuristic mapping: (modify as needed per dataset specifics)
    # Treat zeros as background
    out[mask == 0] = 0
    # Example CSF labels: 1
    out[mask == 1] = 1
    # Example GM labels: 2, 41, 42
    gm_ids = [2, 41, 42]
    for v in gm_ids:
        out[mask == v] = 2
    # Example WM labels: 3, 4, 43
    wm_ids = [3, 4, 43]
    for v in wm_ids:
        out[mask == v] = 3
    # If labels are already 0..3, this is idempotent
    out[np.isin(mask, [1, 2, 3])] = mask[np.isin(mask, [1, 2, 3])]
    return out


def normalize_volume(vol: np.ndarray) -> np.ndarray:
    vmin = vol.min()
    vmax = vol.max()
    if vmax - vmin < 1e-8:
        return np.zeros_like(vol, dtype=np.float32)
    return (vol - vmin) / (vmax - vmin)


def process_ibsr_folder(folder: str, save_dir: str, save_npz: bool = True):
    """Process IBSR folder containing *_ana_strip.nii.gz and *_segTRI_fill_ana.nii.gz files.
    Saves images.npy and masks.npy in save_dir.
    """
    folder = Path(folder)
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    images_paths = sorted(folder.glob("*_ana_strip.nii*"))
    masks_paths = sorted(folder.glob("*_segTRI_fill_ana.nii*"))

    all_imgs = []
    all_masks = []

    for img_path, mask_path in zip(images_paths, masks_paths):
        img = load_nifti(str(img_path))
        mask = load_nifti(str(mask_path))
        mask = map_labels(mask)
        img = normalize_volume(img)

        # extract axial slices (z axis)
        for i in range(img.shape[2]):
            m = mask[:, :, i]
            if m.sum() == 0:
                continue
            all_imgs.append(img[:, :, i])
            all_masks.append(m)

    imgs_np = np.stack([np.asarray(x, dtype=np.float32) for x in all_imgs])
    masks_np = np.stack([np.asarray(x, dtype=np.uint8) for x in all_masks])

    np.save(save_dir / "images.npy", imgs_np)
    np.save(save_dir / "masks.npy", masks_np)
    if save_npz:
        np.savez_compressed(save_dir / "data.npz", images=imgs_np, masks=masks_np)
    print(f"Saved {len(imgs_np)} slices to {save_dir}")


# Example usage (uncomment and set paths before running):
# process_ibsr_folder('/path/to/IBSR_nifti_stripped', './processed_ibsr')

In [ ]:
class IBSR2DSlicesDataset(Dataset):
    def __init__(self, npz_folder: str, transform=None):
        """Load prepared images.npy and masks.npy from npz_folder."""
        self.npz_folder = Path(npz_folder)
        imgs_path = self.npz_folder / "images.npy"
        masks_path = self.npz_folder / "masks.npy"
        if not imgs_path.exists() or not masks_path.exists():
            # try data.npz
            npz_file = self.npz_folder / "data.npz"
            if npz_file.exists():
                data = np.load(npz_file)
                self.images = data["images"]
                self.masks = data["masks"]
            else:
                raise FileNotFoundError("prepared images/masks not found in folder")
        else:
            self.images = np.load(imgs_path)
            self.masks = np.load(masks_path)

        assert len(self.images) == len(self.masks)
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        mask = self.masks[idx]
        # convert to channel-first
        img = np.asarray(img, dtype=np.float32)[None, ...]
        # mask as int64
        mask = np.asarray(mask, dtype=np.int64)
        if self.transform:
            img, mask = self.transform(img, mask)
        # return tensors
        return torch.from_numpy(img), torch.from_numpy(mask)


def collate_pad(batch):
    # batch: list of (img_tensor, mask_tensor)
    imgs = [b[0] for b in batch]
    masks = [b[1] for b in batch]
    # find max H,W
    max_h = max([im.shape[1] for im in imgs])
    max_w = max([im.shape[2] for im in imgs])
    padded_imgs = []
    padded_masks = []
    for im, m in zip(imgs, masks):
        c, h, w = im.shape
        pad_h = max_h - h
        pad_w = max_w - w
        im_p = F.pad(im, (0, pad_w, 0, pad_h))
        m_p = F.pad(m.unsqueeze(0).float(), (0, pad_w, 0, pad_h)).squeeze(0).long()
        padded_imgs.append(im_p)
        padded_masks.append(m_p)
    imgs_tensor = torch.stack(padded_imgs)
    masks_tensor = torch.stack(padded_masks)
    return imgs_tensor, masks_tensor


# quick dataset test (uncomment to run locally)
# ds = IBSR2DSlicesDataset('./processed_ibsr')
# print(len(ds))

In [ ]:
# Metrics: Dice, Jaccard, Precision, Recall (per-class)


def compute_confusion(pred: torch.Tensor, target: torch.Tensor, class_id: int):
    # pred, target are single image tensors (H,W) ints
    pred_c = pred == class_id
    tgt_c = target == class_id
    tp = (pred_c & tgt_c).sum().item()
    fp = (pred_c & (~tgt_c)).sum().item()
    fn = ((~pred_c) & tgt_c).sum().item()
    tn = ((~pred_c) & (~tgt_c)).sum().item()
    return tp, fp, fn, tn


def calculate_metrics_batch(
    pred_logits: torch.Tensor, target: torch.Tensor, num_classes: int = 4
):
    # pred_logits: [B, C, H, W], target: [B, H, W]
    preds = pred_logits.argmax(dim=1) if pred_logits.dim() == 4 else pred_logits
    batch_metrics = {
        "dice": [0] * num_classes,
        "jaccard": [0] * num_classes,
        "precision": [0] * num_classes,
        "recall": [0] * num_classes,
    }
    B = preds.shape[0]
    for c in range(num_classes):
        dices = []
        jacs = []
        precs = []
        recs = []
        for i in range(B):
            tp, fp, fn, tn = compute_confusion(preds[i], target[i], c)
            inter = tp
            denom_dice = 2 * tp + fp + fn
            dice = (2 * inter) / denom_dice if denom_dice > 0 else 1.0
            union = tp + fp + fn
            jaccard = inter / union if union > 0 else 1.0
            precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 1.0
            dices.append(dice)
            jacs.append(jaccard)
            precs.append(precision)
            recs.append(recall)
        batch_metrics["dice"][c] = float(np.mean(dices))
        batch_metrics["jaccard"][c] = float(np.mean(jacs))
        batch_metrics["precision"][c] = float(np.mean(precs))
        batch_metrics["recall"][c] = float(np.mean(recs))
    return batch_metrics


# Example usage: metrics = calculate_metrics_batch(logits, masks)

In [ ]:
# LeCun init and training loop skeleton


def lecun_init_weights(module):
    if isinstance(module, nn.Conv2d):
        fan_in, _ = nn.init._calculate_fan_in_and_fan_out(module.weight)
        std = np.sqrt(1.0 / max(1, fan_in))
        nn.init.normal_(module.weight, mean=0.0, std=std)
        if module.bias is not None:
            nn.init.zeros_(module.bias)


# Training loop (example, may need GPU and proper dataloaders)
def train_model(model, train_loader, val_loader, device, epochs=10, lr=1e-4):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    model.apply(lecun_init_weights)

    history = {"train_loss": [], "val_loss": [], "val_metrics": []}
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for imgs, masks in train_loader:
            imgs = imgs.to(device)
            masks = masks.to(device)
            outputs = model(imgs)
            # outputs: [B, C, H, W]; masks: [B, H, W]
            loss = criterion(outputs, masks)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        avg_train_loss = running_loss / len(train_loader)
        # validation
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_masks = []
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs = imgs.to(device)
                masks = masks.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, masks)
                val_loss += loss.item()
                all_preds.append(outputs.cpu())
                all_masks.append(masks.cpu())
        avg_val_loss = val_loss / len(val_loader)
        # compute metrics on validation (concatenate batches)
        pred_cat = torch.cat([p for p in all_preds], dim=0)
        mask_cat = torch.cat([m for m in all_masks], dim=0)
        metrics = calculate_metrics_batch(pred_cat, mask_cat)
        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["val_metrics"].append(metrics)
        print(
            f"Epoch {epoch}: train_loss={avg_train_loss:.4f}, val_loss={avg_val_loss:.4f}"
        )
        print(f"Val Dice per class: {metrics['dice']}")
    return model, history


# Example: model = MNet(); trained_model, history = train_model(model, train_loader, val_loader, device)